# 05 — Active Aero Drag Delta (ΔCdA)

Measure the drag reduction from 2026 active aero (straight mode vs corner mode)
using coast-down ODE fits from Canada 2026 FP sessions across all drivers.

Key advantage over the 2024 DRS analysis: active aero activates automatically on
designated straights with no gap-to-car-ahead requirement, so there is no slipstream
confound.  Both aero modes occur naturally in the same FP session — straight-mode
segments on the main straight and back straight, corner-mode segments on approaches
to the hairpins and chicane.

Method: fit α (aerodynamic coefficient) separately for each mode per driver, then
compute ΔCdA = 2/ρ × (α_corner − α_straight).  Using per-driver deltas controls for
car-specific setup differences.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

from src.segments import extract_coastdown_segments, segment_active_aero_state
from src.ode_fit import fit_segment, fit_segments_pooled, FitResult
from src.aero_params import air_density, car_mass

os.makedirs('../cache', exist_ok=True)
fastf1.Cache.enable_cache('../cache')

## Extract coast-down segments — all drivers, all FP sessions

In [ ]:
BETA_FIXED = 120.0   # N — same prior as nb02
MIN_R2     = 0.90

# driver → {'straight': [FitResult, ...], 'corner': [FitResult, ...]}
driver_segs: dict[str, dict[str, list[FitResult]]] = defaultdict(lambda: {'straight': [], 'corner': []})
rho_values = []

for session_name in ['FP1', 'FP2', 'FP3']:
    try:
        sess = fastf1.get_session(2026, 'Canada', session_name)
        sess.load(telemetry=True, weather=True)
    except Exception as e:
        print(f'Could not load {session_name}: {e}')
        continue

    rho_s = air_density(sess.weather_data['AirTemp'].mean(), sess.weather_data['Pressure'].mean())
    rho_values.append(rho_s)
    n_before = sum(len(v['straight']) + len(v['corner']) for v in driver_segs.values())

    for driver in sess.drivers:
        try:
            laps = sess.laps.pick_drivers(driver)
            laps = laps[laps['LapNumber'] > 1]
        except Exception:
            continue
        for _, lap in laps.iterrows():
            lap_num = int(lap['LapNumber'])
            m = car_mass(lap_num)
            try:
                tel = lap.get_telemetry()
            except Exception:
                continue
            segs = extract_coastdown_segments(
                tel, min_duration=0.5, min_speed_kmh=120.0, throttle_threshold=5.0,
            )
            for seg in segs:
                straight_mode = segment_active_aero_state(seg)
                r = fit_segment(seg, m, rho_s, straight_mode, lap_num, beta_fixed=BETA_FIXED)
                if r is not None and r.r2 >= MIN_R2:
                    key = 'straight' if straight_mode else 'corner'
                    driver_segs[driver][key].append(r)

    n_after = sum(len(v['straight']) + len(v['corner']) for v in driver_segs.values())
    print(f'{session_name}: +{n_after - n_before} valid segments')

rho = float(np.mean(rho_values))

total_straight = sum(len(v['straight']) for v in driver_segs.values())
total_corner   = sum(len(v['corner'])   for v in driver_segs.values())
print(f'\nTotal:  straight mode = {total_straight},  corner mode = {total_corner}')
print(f'Drivers with ≥1 segment: {len(driver_segs)}')
print(f'Mean air density: {rho:.4f} kg/m³')

## Per-driver α by mode

In [ ]:
driver_stats = []

for driver, modes in driver_segs.items():
    straight_rs = modes['straight']
    corner_rs   = modes['corner']
    if not straight_rs and not corner_rs:
        continue
    row = {'driver': driver,
           'n_straight': len(straight_rs),
           'n_corner':   len(corner_rs)}
    if straight_rs:
        row['alpha_straight'] = float(np.median([r.alpha for r in straight_rs]))
    else:
        row['alpha_straight'] = np.nan
    if corner_rs:
        row['alpha_corner'] = float(np.median([r.alpha for r in corner_rs]))
    else:
        row['alpha_corner'] = np.nan
    driver_stats.append(row)

stats_df = pd.DataFrame(driver_stats).sort_values('n_straight', ascending=False)
stats_df['delta_alpha'] = stats_df['alpha_corner'] - stats_df['alpha_straight']
stats_df['delta_CdA']   = 2.0 * stats_df['delta_alpha'] / rho

print(stats_df[['driver','n_straight','n_corner','alpha_straight','alpha_corner','delta_CdA']]
      .to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Drivers with both modes — these drive the pooled estimate
both_mask = stats_df['alpha_straight'].notna() & stats_df['alpha_corner'].notna()
stats_both = stats_df[both_mask]
print(f'\nDrivers with segments in both modes: {len(stats_both)}')

## Pooled ΔCdA estimate with bootstrap CI

In [ ]:
if len(stats_both) == 0:
    print('No drivers with segments in both modes — cannot compute ΔCdA.')
    DELTA_CDA_POOLED = np.nan
    DELTA_CDA_STD    = np.nan
else:
    # Weighted mean: weight by min(n_straight, n_corner) — proxy for reliability
    w = stats_both[['n_straight','n_corner']].min(axis=1).values.astype(float)
    deltas = stats_both['delta_CdA'].values
    DELTA_CDA_POOLED = float(np.average(deltas, weights=w))

    # Bootstrap: resample drivers with replacement, recompute weighted mean
    rng = np.random.default_rng(42)
    boot = []
    for _ in range(10_000):
        idx = rng.integers(0, len(deltas), size=len(deltas))
        d_b = deltas[idx]
        w_b = w[idx]
        boot.append(float(np.average(d_b, weights=w_b)))
    ci = np.percentile(boot, [5, 95])
    DELTA_CDA_STD = float(np.std(boot))

    print(f'=== Active Aero ΔCdA ===')
    print(f'  Pooled ΔCdA (straight − corner) = {DELTA_CDA_POOLED:.3f} ± {DELTA_CDA_STD:.3f} m²')
    print(f'  90% CI: [{ci[0]:.3f}, {ci[1]:.3f}] m²')
    print(f'  Based on {len(stats_both)} drivers with both straight and corner mode segments')
    print(f'  Expected (2026 active aero): −0.5 to −1.2 m²')
    print()
    print('  Note: engine braking affects both modes differently (different speeds/gears).')
    print('  The delta partially cancels shared biases but residual systematic error remains.')

## Visualise

In [ ]:
if len(stats_both) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: per-driver ΔCdA
    ax = axes[0]
    drivers_sorted = stats_both.sort_values('delta_CdA')
    colors = ['tomato' if d < 0 else 'steelblue' for d in drivers_sorted['delta_CdA']]
    ax.barh(drivers_sorted['driver'], drivers_sorted['delta_CdA'], color=colors)
    ax.axvline(DELTA_CDA_POOLED, color='k', linestyle='--',
               label=f'pooled = {DELTA_CDA_POOLED:.3f} m²')
    ax.axvline(0, color='gray', linewidth=0.8)
    ax.set_xlabel('ΔCdA (straight − corner) m²')
    ax.set_title('Per-driver active aero drag delta')
    ax.legend()

    # Right: bootstrap distribution
    ax = axes[1]
    ax.hist(boot, bins=50, edgecolor='black', color='orchid')
    ax.axvline(DELTA_CDA_POOLED, color='r', linestyle='--',
               label=f'pooled = {DELTA_CDA_POOLED:.3f} m²')
    ax.axvline(ci[0], color='gray', linestyle=':', label='90% CI')
    ax.axvline(ci[1], color='gray', linestyle=':')
    ax.axvline(0, color='k', linestyle=':', linewidth=0.8)
    ax.set_xlabel('ΔCdA (m²)')
    ax.set_ylabel('bootstrap count')
    ax.set_title('Bootstrap distribution of pooled ΔCdA')
    ax.legend()

    plt.tight_layout()
    plt.savefig('../results/figures/05_active_aero_delta.png', dpi=150)
    plt.show()

    # α scatter by mode (all drivers)
    all_straight_alphas = [r.alpha for d in driver_segs.values() for r in d['straight']]
    all_corner_alphas   = [r.alpha for d in driver_segs.values() for r in d['corner']]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(all_corner_alphas,   bins=20, alpha=0.6, color='steelblue', label='corner mode')
    ax.hist(all_straight_alphas, bins=20, alpha=0.6, color='tomato',    label='straight mode')
    if all_corner_alphas:
        ax.axvline(np.median(all_corner_alphas),   color='steelblue', linestyle='--')
    if all_straight_alphas:
        ax.axvline(np.median(all_straight_alphas), color='tomato',    linestyle='--')
    ax.set_xlabel('α (N·s²/m²)')
    ax.set_ylabel('count')
    ax.set_title('α distribution by active aero mode — all drivers, Canada 2026 FP')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/figures/05_alpha_by_mode.png', dpi=150)
    plt.show()
else:
    print('Insufficient data for plots.')